# Fig 6 - RQ1: explicit response vs. implicit expected score, per model
For each model, scatter of explicit (discrete) response vs. implicit (continuous) expected score, with Pearson r and mean signed difference reported in each panel.

## Load data

In [ ]:
"""Load the merged data using the project's data_loading module."""
from data_loading import load_data

df = load_data()
df.head()


## Per-model correlations between explicit and implicit

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr

models = [
    ("Qwen 2.5-72B Instruct",  "qwen_response",            "qwen_expected"),
    ("Llama 3.3-70B Instruct", "llama_instruct_response", "llama_instruct_expected"),
    ("Llama 3.1-70B Base",     "llama_base_response",     "llama_base_expected"),
]

rows = []
for name, exp_col, imp_col in models:
    sub = df[[exp_col, imp_col]].dropna()
    r,   _ = pearsonr(sub[exp_col], sub[imp_col])
    rho, _ = spearmanr(sub[exp_col], sub[imp_col])
    diff = sub[imp_col] - sub[exp_col]
    rows.append({
        "Model": name, "N": len(sub),
        "Pearson r": r, "Spearman rho": rho,
        "Mean diff (imp - exp)": diff.mean(),
        "Mean abs diff": diff.abs().mean(),
    })

pd.DataFrame(rows)


## Build the 3-panel scatter plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

panels = [
    ("Qwen 2.5-72B Instruct",  "qwen_response",            "qwen_expected",            "#9467bd"),
    ("Llama 3.3-70B Instruct", "llama_instruct_response", "llama_instruct_expected",   "#2ca02c"),
    ("Llama 3.1-70B Base",     "llama_base_response",     "llama_base_expected",       "#d62728"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), dpi=150, sharey=True)
rng = np.random.default_rng(0)
for ax, (name, exp_col, imp_col, color) in zip(axes, panels):
    sub = df[[exp_col, imp_col]].dropna()
    x = sub[exp_col].values
    y = sub[imp_col].values
    x_j = x + rng.uniform(-0.18, 0.18, size=len(x))
    ax.scatter(x_j, y, s=4, alpha=0.08, color=color, edgecolor="none")
    ax.plot([0, 5], [0, 5], color="black", linestyle="--", linewidth=1, alpha=0.6, label="y = x")
    r = np.corrcoef(x, y)[0, 1]
    mean_d = (y - x).mean()
    ax.set_title(f"{name}\nPearson r = {r:.3f}    mean(impl-expl) = {mean_d:+.3f}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Explicit response (discrete, 0-5)", fontsize=11)
    ax.set_xlim(-0.6, 5.6)
    ax.set_ylim(-0.2, 5.2)
    ax.set_xticks([0, 1, 2, 3, 4, 5])
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="upper left", fontsize=9)

axes[0].set_ylabel("Implicit expected score (continuous, 0-5)", fontsize=11)
fig.suptitle("Explicit responses vs. implicit (token-probability) expected scores, per model",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig6_explicit_vs_implicit.png", dpi=200, bbox_inches="tight")
plt.show()
